# Image-Only Fraud Detection — EfficientNet-B0
Two experiments:
- **A_weighted** — BCEWithLogitsLoss + pos_weight
- **B_unweighted** — BCEWithLogitsLoss no weighting

Structure mirrors notebooks 19 (multimodal).
All functions from `efficientnet_utils.py` and `mm_image_utils.py`.


## 0 · Imports & reproducibility

In [ ]:
import sys, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.metrics import (roc_curve, auc as sk_auc,
                             precision_recall_curve, average_precision_score)

sys.path.insert(0, r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\src")

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

from efficientnet_utils import (
    build_model, unfreeze_backbone,
    fit_model, run_test_evaluation, predict_probs,
    compute_metrics, tune_threshold,
    build_train_transform, build_val_transform,
    GradCAM,
)
from mm_image_utils import (
    load_mm_splits, build_mm_dataloaders,
    make_loss_fn, subgroup_by_combo,
)

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

SEED   = 42
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}  |  PyTorch {torch.__version__}")


## 1 · Config

In [ ]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "data" / "processed"

BACKBONE = "efficientnet_b0"

TRAIN_CSV = DATA_DIR / "mm_train_mixed_all_group_test.csv"
VAL_CSV   = DATA_DIR / "mm_val_mixed_all_group_test.csv"
TEST_CSV  = DATA_DIR / "mm_test_mixed_all_group_test.csv"

BATCH_SIZE  = 32
NUM_WORKERS = 0
STAGE1_EPOCHS   = 5;  STAGE1_LR = 1e-3;  STAGE1_PATIENCE = 5
STAGE2_EPOCHS   = 20; BACKBONE_LR = 1e-5; HEAD_LR = 1e-4; STAGE2_PATIENCE = 6
DROPOUT = 0.4

RESULTS_BASE = PROJECT_ROOT / "notebook" / "results" / f"mm_image_{BACKBONE}"
EXP_DIRS = {
    "A_weighted":   RESULTS_BASE / "A_weighted",
    "B_unweighted": RESULTS_BASE / "B_unweighted",
}
for d in EXP_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"Backbone    : {BACKBONE}")
print(f"Results base: {RESULTS_BASE}")

COMBO_COLOURS = {"tab0_img0":"#4C72B0","tab1_img1":"#C44E52",
                 "tab1_img0":"#DD8452","tab0_img1":"#55A868"}


## 2 · Load data

In [ ]:
from torch import nn

train_df, val_df, test_df = load_mm_splits(TRAIN_CSV, VAL_CSV, TEST_CSV)

train_loader, val_loader, test_loader = build_mm_dataloaders(
    train_df, val_df, test_df,
    train_transform=build_train_transform(),
    val_transform=build_val_transform(),
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    use_sampler=False,
)
print(f"Batches — train:{len(train_loader)} val:{len(val_loader)} test:{len(test_loader)}")

LOSS_FNS = {
    "A_weighted":   make_loss_fn(train_df, device),      # pos_weight
    "B_unweighted": nn.BCEWithLogitsLoss().to(device),   # no weighting
}
print("Loss functions ready:", list(LOSS_FNS))


## 3 · Dataset imbalance analysis
Mirrors the multimodal notebook — shows why weighted loss matters.


In [ ]:
COMBO_COLOURS = {"tab0_img0":"#4C72B0","tab1_img1":"#C44E52",
                 "tab1_img0":"#DD8452","tab0_img1":"#55A868"}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, df) in zip(axes[:2], [("Train", train_df), ("Test", test_df)]):
    counts = df["final_label"].value_counts().sort_index()
    bars = ax.bar(["Legitimate (0)", "Fraud (1)"], counts.values,
                  color=["#4C72B0","#DD8452"], edgecolor="white", width=0.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
                str(val), ha="center", va="bottom", fontsize=10)
    ratio = counts.get(1,0) / max(counts.get(0,1), 1)
    ax.set_title(f"{name}  ratio 1:{ratio:.2f}"); ax.set_ylabel("Count")
    ax.spines[["top","right"]].set_visible(False)

if "img_source_dataset" in train_df.columns:
    pd.crosstab(train_df["img_source_dataset"],
                train_df["combo_type"]).plot(
        kind="bar", ax=axes[2],
        color=list(COMBO_COLOURS.values()), edgecolor="white")
    axes[2].set_title("Train — source × combo")
    axes[2].set_xlabel(""); axes[2].tick_params(axis="x", rotation=0)
    axes[2].spines[["top","right"]].set_visible(False)

fig.suptitle("Image-only dataset imbalance", fontsize=12)
fig.tight_layout()
fig.savefig(RESULTS_BASE / "imbalance_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

n0 = (train_df["final_label"]==0).sum()
n1 = (train_df["final_label"]==1).sum()
print(f"pos_weight = {n0}/{n1} = {n0/n1:.3f}")


## 4 · Training helper

In [ ]:
def run_experiment(exp_id):
    """
    Two-stage train + evaluation for one experiment.
    Mirrors run_experiment() from notebook 19.
    h_s1=None pattern avoids NameError when stage 1 is cached.
    """
    results_dir = EXP_DIRS[exp_id]
    loss_fn     = LOSS_FNS[exp_id]

    print(f"\n{'='*60}")
    print(f"EXPERIMENT {exp_id}")
    print(f"  backbone : {BACKBONE}")
    print(f"  dropout  : {DROPOUT}")
    print(f"  loss     : {type(loss_fn).__name__}")
    print(f"{'='*60}")

    # Stage 1 — frozen backbone
    h_s1 = None                             # ← same pattern as notebook 19
    if (results_dir / "stage1_best.pt").exists():
        print("Stage 1 already trained — skipping.")
        model = build_model(pretrained=False, freeze_backbone=True,
                            dropout=DROPOUT).to(device)
        model.load_state_dict(
            torch.load(results_dir / "stage1_best.pt", map_location=device))
    else:
        model = build_model(pretrained=True, freeze_backbone=True,
                            dropout=DROPOUT).to(device)
        opt1 = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=STAGE1_LR, weight_decay=1e-4)
        model, h_s1 = fit_model(
            model=model, train_loader=train_loader, val_loader=val_loader,
            loss_fn=loss_fn, optimizer=opt1, device=device,
            epochs=STAGE1_EPOCHS, early_stopping_patience=STAGE1_PATIENCE,
            monitor_metric="val_roc_auc",
            model_save_path=results_dir / "stage1_best.pt",
        )
        display(h_s1.round(4))

    # Stage 2 — full fine-tuning
    if (results_dir / "stage2_best.pt").exists():
        print("Stage 2 already trained — skipping.")
        unfreeze_backbone(model)
        model.load_state_dict(
            torch.load(results_dir / "stage2_best.pt", map_location=device))
    else:
        unfreeze_backbone(model)
        bp = [p for n, p in model.named_parameters() if "classifier" not in n]
        hp = list(model.classifier.parameters())
        opt2 = torch.optim.Adam(
            [{"params": bp, "lr": BACKBONE_LR},
             {"params": hp, "lr": HEAD_LR}],
            weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt2, T_max=STAGE2_EPOCHS, eta_min=1e-7)
        model, h_s2 = fit_model(
            model=model, train_loader=train_loader, val_loader=val_loader,
            loss_fn=loss_fn, optimizer=opt2, device=device,
            epochs=STAGE2_EPOCHS, early_stopping_patience=STAGE2_PATIENCE,
            monitor_metric="val_roc_auc", scheduler=sched, freeze_bn=True,
            model_save_path=results_dir / "stage2_best.pt",
        )
        # h_s1 may be None if stage 1 was cached — safe concat
        hist = (pd.concat([h_s1, h_s2], ignore_index=True)
                if h_s1 is not None else h_s2.copy())
        hist["epoch_global"] = range(1, len(hist) + 1)
        hist.to_csv(results_dir / "training_history.csv", index=False)
        display(h_s2.round(4))

    model.eval()

    # Threshold tuning on validation set
    val_y, val_p     = predict_probs(model, val_loader, device)
    best_t, sweep_df = tune_threshold(val_y, val_p, metric="f1")
    sweep_df.to_csv(results_dir / "threshold_sweep.csv", index=False)

    # Test evaluation
    test_metrics = run_test_evaluation(
        model=model, test_loader=test_loader,
        loss_fn=loss_fn, device=device, threshold=best_t)

    y_true, y_prob = predict_probs(model, test_loader, device)
    pred_df = test_df.reset_index(drop=True).copy()
    pred_df["y_true"] = y_true
    pred_df["y_prob"] = y_prob
    pred_df["y_pred"] = (pred_df["y_prob"] >= best_t).astype(int)
    pred_df.to_csv(results_dir / "test_predictions.csv", index=False)

    sg = subgroup_by_combo(pred_df, best_t, exp_id)
    sg.to_csv(results_dir / "subgroup_combo_type.csv", index=False)

    def get_f1(combo):
        r = sg[sg["combo_type"] == combo]
        return r["f1"].values[0] if len(r) > 0 else float("nan")

    print(f"  tab0_img0 F1 : {get_f1('tab0_img0'):.3f}  (expected 0.000 — no image signal)")
    print(f"  tab1_img0 F1 : {get_f1('tab1_img0'):.3f}  (hard case — tab fraud, normal img)")

    return {"id": exp_id, "label": exp_id, "threshold": best_t,
            "metrics": test_metrics, "pred_df": pred_df,
            "sg": sg, "y_true": y_true, "y_prob": y_prob}


---
## 5 · Experiment A — Weighted loss
BCEWithLogitsLoss with pos_weight to up-weight the minority class.


In [ ]:
res_A = run_experiment("A_weighted")

---
## 6 · Experiment B — Unweighted loss
BCEWithLogitsLoss with no correction — baseline.


In [ ]:
res_B = run_experiment("B_unweighted")

---
## 7 · Load all results
Run this cell to reload from disk without retraining.
Mirrors `load_result()` from notebook 19.


In [ ]:
# Check all experiment folders have required files before loading
for exp_id, path in EXP_DIRS.items():
    print("\n", exp_id)
    print("Folder exists            :", path.exists())
    print("test_predictions.csv     :", (path / "test_predictions.csv").exists())
    print("subgroup_combo_type.csv  :", (path / "subgroup_combo_type.csv").exists())
    print("threshold_sweep.csv      :", (path / "threshold_sweep.csv").exists())
    print("training_history.csv     :", (path / "training_history.csv").exists())


In [ ]:
def load_result(exp_id):
    results_dir = EXP_DIRS[exp_id]
    pred_df = pd.read_csv(results_dir / "test_predictions.csv")
    thresh  = float(
        pd.read_csv(results_dir / "threshold_sweep.csv")
        .pipe(lambda d: d.loc[d["f1"].idxmax(), "threshold"])
    )
    metrics = compute_metrics(pred_df["y_true"].values,
                              pred_df["y_prob"].values, thresh)
    sg      = pd.read_csv(results_dir / "subgroup_combo_type.csv")
    return {"id": exp_id, "label": exp_id, "threshold": thresh,
            "metrics": metrics, "pred_df": pred_df, "sg": sg,
            "y_true": pred_df["y_true"].values,
            "y_prob": pred_df["y_prob"].values}

res_A = load_result("A_weighted")
res_B = load_result("B_unweighted")

results = [res_A, res_B]
labels  = ["A: Weighted", "B: Unweighted"]
colours = ["#4C72B0", "#DD8452"]
print("All results loaded.")


## 8 · Training curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for row, (exp_id, label) in enumerate(zip(EXP_DIRS.keys(), labels)):
    hist_path = EXP_DIRS[exp_id] / "training_history.csv"
    if not hist_path.exists():
        print(f"No history for {exp_id}"); continue
    hist = pd.read_csv(hist_path)
    if "epoch_global" not in hist.columns:
        hist["epoch_global"] = range(1, len(hist) + 1)
    ep      = hist["epoch_global"]
    n_s1    = STAGE1_EPOCHS
    valid   = hist["val_f1"].dropna()
    best_ep = ep.iloc[valid.idxmax()] if not valid.empty else None

    axes[row, 0].plot(ep, hist["train_loss"], label="train", color="#4C72B0")
    axes[row, 0].plot(ep, hist["val_loss"],   label="val",   color="#DD8452")
    axes[row, 0].set_title(f"Loss — {label}"); axes[row, 0].legend(fontsize=8)

    axes[row, 1].plot(ep, hist["val_f1"],      label="F1",      color="#55A868")
    axes[row, 1].plot(ep, hist["val_roc_auc"], label="ROC-AUC", color="#C44E52")
    axes[row, 1].set_ylim(0.3, 1.01)
    axes[row, 1].set_title(f"Val metrics — {label}"); axes[row, 1].legend(fontsize=8)

    for ax in axes[row]:
        ax.set_xlabel("Epoch")
        ax.axvline(x=n_s1 + 0.5, color="gray", linestyle="--", alpha=0.6, label="stage 1→2")
        if best_ep is not None:
            ax.axvline(x=best_ep, color="red", linestyle=":", alpha=0.6, label="best val F1")
        ax.spines[["top","right"]].set_visible(False)

fig.suptitle(f"Image-only ({BACKBONE}) — training curves", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_BASE / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 9 · Overall metrics comparison

In [ ]:
cmp = pd.DataFrame([
    {"experiment": r["id"],
     "loss":       "Weighted" if "weighted" in r["id"] else "Unweighted",
     "threshold":  r["threshold"],
     "accuracy":   r["metrics"]["accuracy"],
     "precision":  r["metrics"]["precision"],
     "recall":     r["metrics"]["recall"],
     "f1":         r["metrics"]["f1"],
     "roc_auc":    r["metrics"]["roc_auc"]}
    for r in results
])
display(cmp.round(4))
cmp.to_csv(RESULTS_BASE / "comparison.csv", index=False)

metrics_plot = ["accuracy","precision","recall","f1","roc_auc"]
x = np.arange(len(metrics_plot)); width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
for i, (r, label, colour) in enumerate(zip(results, labels, colours)):
    ax.bar(x + (i - 0.5)*width, [r["metrics"][m] for m in metrics_plot],
           width, label=label, color=colour, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(["Accuracy","Precision","Recall","F1","ROC-AUC"])
ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
ax.set_title(f"Image-only ({BACKBONE}) — Weighted vs Unweighted")
ax.legend(fontsize=9); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS_BASE / "comparison_overall.png", dpi=150, bbox_inches="tight")
plt.show()


## 10 · PR and ROC curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for r, label, colour in zip(results, labels, colours):
    prec, rec, _ = precision_recall_curve(r["y_true"], r["y_prob"])
    ap = average_precision_score(r["y_true"], r["y_prob"])
    axes[0].plot(rec, prec, color=colour, lw=2, label=f"{label}  AP={ap:.4f}")

    fpr, tpr, _ = roc_curve(r["y_true"], r["y_prob"])
    axes[1].plot(fpr, tpr, color=colour, lw=2,
                 label=f"{label}  AUC={sk_auc(fpr,tpr):.4f}")

axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("PR curves"); axes[0].legend(fontsize=9)
axes[0].spines[["top","right"]].set_visible(False)
axes[1].plot([0,1],[0,1],"k--",lw=1)
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("ROC curves"); axes[1].legend(fontsize=9)
axes[1].spines[["top","right"]].set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS_BASE / "pr_roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 11 · Subgroup analysis by combo_type
Key diagnostic: image-only models cannot detect tabular fraud (tab1_img0 F1 ≈ 0.30).


In [ ]:
COMBO_COLOURS = {"tab0_img0":"#4C72B0","tab1_img1":"#C44E52",
                 "tab1_img0":"#DD8452","tab0_img1":"#55A868"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, r, label in zip(axes, results, labels):
    sg   = r["sg"]
    clrs = [COMBO_COLOURS.get(c,"gray") for c in sg["combo_type"]]
    bars = ax.bar(sg["combo_type"], sg["f1"], color=clrs,
                  edgecolor="white", width=0.5)
    for bar, val in zip(bars, sg["f1"]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylim(0, 1.25); ax.set_title(label, fontsize=10)
    ax.axhline(0.5, color="red", linestyle="--", alpha=0.4)
    ax.set_ylabel("F1"); ax.tick_params(axis="x", rotation=15)
    ax.spines[["top","right"]].set_visible(False)

fig.legend(handles=[
    Patch(color="#4C72B0", label="tab0_img0 → label 0 (legitimate)"),
    Patch(color="#C44E52", label="tab1_img1 → label 1 (both fraud)"),
    Patch(color="#DD8452", label="tab1_img0 → label 1 (tab only) ← key test"),
    Patch(color="#55A868", label="tab0_img1 → label 1 (img only)"),
], loc="lower center", ncol=4, bbox_to_anchor=(0.5,-0.08), fontsize=9)
fig.suptitle(f"Image-only ({BACKBONE}) — Combo F1", fontsize=12)
fig.tight_layout()
fig.savefig(RESULTS_BASE / "subgroup_combo.png", dpi=150, bbox_inches="tight")
plt.show()

# Comparison table
sg_cmp = pd.concat([
    r["sg"].set_index("combo_type")[["f1"]].rename(columns={"f1": r["id"]})
    for r in results
], axis=1).round(3)
display(sg_cmp)
sg_cmp.to_csv(RESULTS_BASE / "subgroup_comparison.csv")


## 12 · Score distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, r, label in zip(axes, results, labels):
    ax.hist(r["y_prob"][r["y_true"]==0], bins=40, alpha=0.65,
            color="#4C72B0", label="Legitimate", density=True)
    ax.hist(r["y_prob"][r["y_true"]==1], bins=40, alpha=0.65,
            color="#DD8452", label="Fraud", density=True)
    ax.axvline(x=r["threshold"], color="red", linestyle="--",
               label=f"t={r['threshold']:.2f}")
    ax.set_title(label); ax.set_xlabel("Predicted fraud probability")
    ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)
fig.suptitle(f"Score distributions — {BACKBONE}", fontsize=12)
fig.tight_layout()
fig.savefig(RESULTS_BASE / "score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


## 13 · Confusion matrices

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, r, label in zip(axes, results, labels):
    cm = confusion_matrix(r["y_true"], (r["y_prob"] >= r["threshold"]).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                cbar=False, linewidths=0.5, linecolor="white")
    tp, fp, fn = cm[1,1], cm[0,1], cm[1,0]
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_xticklabels(["Non-fraud","Fraud"])
    ax.set_yticklabels(["Non-fraud","Fraud"], rotation=0)
    ax.set_title(f"{label}  |  FN={fn}  FP={fp}", fontsize=10)
fig.suptitle(f"Confusion matrices — {BACKBONE}", fontsize=12)
fig.tight_layout()
fig.savefig(RESULTS_BASE / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


## 14 · Final summary table

In [ ]:
def gf1(sg, combo):
    r = sg[sg["combo_type"] == combo]
    return f"{r['f1'].values[0]:.3f}" if len(r) > 0 else "N/A"

fig, ax = plt.subplots(figsize=(14, 2.5))
ax.axis("off")
col_labels = ["Experiment","Loss","Threshold",
              "Accuracy","Precision","Recall","F1","ROC-AUC",
              "tab0_img0 F1","tab1_img0 F1"]
cell_data = [
    [r["id"],
     "Weighted" if "weighted" in r["id"] else "Unweighted",
     f"{r['threshold']:.2f}",
     f"{r['metrics']['accuracy']:.4f}",
     f"{r['metrics']['precision']:.4f}",
     f"{r['metrics']['recall']:.4f}",
     f"{r['metrics']['f1']:.4f}",
     f"{r['metrics']['roc_auc']:.4f}",
     gf1(r["sg"],"tab0_img0"),
     gf1(r["sg"],"tab1_img0")]
    for r in results
]
tbl = ax.table(cellText=cell_data, colLabels=col_labels,
               cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.05, 2.0)
ax.set_title(f"Image-only ({BACKBONE}) — Final Summary  |  seed={SEED}",
             fontsize=12, pad=15)
fig.tight_layout()
fig.savefig(RESULTS_BASE / "final_summary.png", dpi=150, bbox_inches="tight")
plt.show()

best = max(results, key=lambda r: r["metrics"]["f1"])
print(f"Best experiment : {best['id']}")
print(f"  F1            : {best['metrics']['f1']:.4f}")
print(f"  ROC-AUC       : {best['metrics']['roc_auc']:.4f}")
print(f"  tab1_img0 F1  : {gf1(best['sg'],'tab1_img0')}")
print(f"\nAll results saved to: {RESULTS_BASE}")


## 15 · Per-combo diagnostics
Mirrors cells 36–39 from notebook 19.

In [ ]:
# Per-combo detailed diagnostics — mirrors notebook 19
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

for combo in ["tab0_img0", "tab0_img1", "tab1_img0", "tab1_img1"]:
    g = res_A["pred_df"][res_A["pred_df"]["combo_type"] == combo]
    y_true = g["y_true"]; y_pred = g["y_pred"]
    print("\n", combo)
    print("n:", len(g))
    print("true label counts:"); print(y_true.value_counts())
    print("pred label counts:"); print(y_pred.value_counts())
    print("accuracy:", accuracy_score(y_true, y_pred))
    print("F1 fraud  label 1:", f1_score(y_true, y_pred, pos_label=1, zero_division=0))
    print("F1 legit  label 0:", f1_score(y_true, y_pred, pos_label=0, zero_division=0))


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def combo_metrics_fixed(res):
    """
    Computes per-combo metrics correctly.
    tab0_img0 is legitimate-only  → use pos_label=0 (legit F1, FPR).
    tab1_img0 is tabular-fraud    → use pos_label=1 (fraud F1).
    Mirrors combo_metrics_fixed() from notebook 19.
    """
    df   = res["pred_df"].copy()
    rows = {}
    for combo, g in df.groupby("combo_type"):
        y_true = g["y_true"]; y_pred = g["y_pred"]
        acc    = accuracy_score(y_true, y_pred)
        if combo == "tab0_img0":
            legit_f1 = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
            fp       = ((y_true == 0) & (y_pred == 1)).sum()
            n_legit  = (y_true == 0).sum()
            fpr      = fp / n_legit if n_legit > 0 else float("nan")
            rows["tab0_img0_acc"]       = acc
            rows["tab0_img0_legit_f1"]  = legit_f1
            rows["tab0_img0_fpr"]       = fpr
        elif combo == "tab1_img0":
            rows["tab1_img0_f1"]  = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
            rows["tab1_img0_acc"] = acc
        else:
            rows[f"{combo}_acc"] = acc
    return rows


In [ ]:
summary_rows = []
for res, loss in [(res_A, "Weighted"), (res_B, "Unweighted")]:
    m     = res["metrics"]
    combo = combo_metrics_fixed(res)
    summary_rows.append({
        "Experiment":          res["id"],
        "Loss":                loss,
        "Threshold":           res["threshold"],
        "Accuracy":            m.get("accuracy"),
        "Precision":           m.get("precision"),
        "Recall":              m.get("recall"),
        "F1":                  m.get("f1"),
        "ROC-AUC":             m.get("roc_auc"),
        "tab0_img0 Legit F1":  combo.get("tab0_img0_legit_f1"),
        "tab0_img0 FPR":       combo.get("tab0_img0_fpr"),
        "tab1_img0 Fraud F1":  combo.get("tab1_img0_f1"),
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)


In [ ]:
num_cols = ["Threshold","Accuracy","Precision","Recall","F1","ROC-AUC",
            "tab0_img0 Legit F1","tab0_img0 FPR","tab1_img0 Fraud F1"]
summary_df_round = summary_df.copy()
summary_df_round[num_cols] = summary_df_round[num_cols].round(4)
print(summary_df_round.to_string(index=False))
summary_df_round.to_csv(RESULTS_BASE / "summary_fixed.csv", index=False)
